# Task 4 — Model Integration, Evaluation & System Simulation

**Gaju — Student D — Formative 2: Multimodal Data Preprocessing**

This notebook wires the three trained models into the sequential gate from the assignment diagram:

1. Face check → deny if unrecognized
2. Product prediction (withheld)
3. Voice check → deny if mismatch
4. Display recommended product

Artifacts expected under `data/processed/`:
- `facial_recognition_model.json` / `.pkl`
- `voice_verification_model.json` / `.pkl`
- `product_recommendation_model.json` / `.pkl`
- `merged_dataset.csv`, `image_features.csv`, `audio_features.csv`

## 0. Setup

In [1]:
import json
import os
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", None)
print("Working directory:", ROOT)

Working directory: /Users/FullTimeStudio/Dev/Multimodal-Data-Preprocessing/formative2-multimodal-preprocessing


## 1. Confirm processed artifacts

Team pipelines already produced the merged tabular set, image/audio feature CSVs, and trained model files. This cell only verifies they are present (no mock data generation).

In [1]:
required = [
    ROOT / "data" / "processed" / "merged_dataset.csv",
    ROOT / "data" / "processed" / "image_features.csv",
    ROOT / "data" / "processed" / "audio_features.csv",
    ROOT / "data" / "processed" / "facial_recognition_model.json",
    ROOT / "data" / "processed" / "voice_verification_model.json",
    ROOT / "data" / "processed" / "product_recommendation_model.json",
    ROOT / "data" / "demo" / "images" / "unauthorized_face_01.jpg",
    ROOT / "data" / "demo" / "audio" / "yes_approve.wav",
]

missing = [str(p.relative_to(ROOT)) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required artifacts:\n- " + "\n- ".join(missing))

audio_members = sorted(
    p.name for p in (ROOT / "data" / "raw" / "audio").iterdir() if p.is_dir() and p.name != ".gitkeep"
)
image_members = sorted(
    p.name for p in (ROOT / "data" / "raw" / "images").iterdir() if p.is_dir() and p.name != ".gitkeep"
)
print("Audio members:", audio_members)
print("Image members:", image_members)
assert audio_members == image_members == ["andrew", "divine", "gaju", "honour"]
print("All required artifacts present.")

Audio members: ['andrew', 'divine', 'gaju', 'honour']
Image members: ['andrew', 'divine', 'gaju', 'honour']
All required artifacts present.


## 2. Model comparison table

Pulls accuracy / F1 / loss from each model's JSON report (same numbers used in `docs/report.md`).

In [1]:
rows = []
for name, path in [
    ("Facial Recognition", ROOT / "data" / "processed" / "facial_recognition_model.json"),
    ("Voice Verification", ROOT / "data" / "processed" / "voice_verification_model.json"),
    ("Product Recommendation", ROOT / "data" / "processed" / "product_recommendation_model.json"),
]:
    report = json.loads(path.read_text())
    metrics = report["selected_model_metrics"]
    rows.append({
        "model": name,
        "selected": report["selected_model"],
        "accuracy": round(metrics["accuracy"], 3),
        "f1": round(metrics["f1_score"], 3),
        "log_loss": round(metrics["log_loss"], 3),
    })

comparison = pd.DataFrame(rows)
comparison

,model,selected,accuracy,f1,log_loss
0,Facial Recognition,logistic_regression,0.900,0.900,0.242
1,Voice Verification,logistic_regression,0.979,0.960,0.182
2,Product Recommendation,logistic_regression,0.500,0.435,2.375


## 3. Interactive CLI (optional)

Uncomment to prompt for image path, `customer_id`, and audio path.

In [ ]:
# from app.cli_app import interactive_main
# interactive_main()

## 4. Required simulation scenarios

Uses the same paths as `python -m app.simulate_scenarios`:

1. Unauthorized image → deny at Stage 1
2. Authorized face + unauthorized voice → deny at Stage 3
3. Full authorized path → grant at Stage 4

In [1]:
from app.cli_app import run_transaction

AUDIO_DIR = ROOT / "data" / "raw" / "audio"
IMAGE_DIR = ROOT / "data" / "raw" / "images"
DEMO_DIR = ROOT / "data" / "demo"

scenarios = [
    {
        "name": "Unauthorized image attempt",
        "image_path": str(DEMO_DIR / "images" / "unauthorized_face_01.jpg"),
        "customer_id": 100,
        "audio_path": str(AUDIO_DIR / "andrew" / "andrew_yes_approve.wav"),
        "expected": "denied",
    },
    {
        "name": "Authorized face, unauthorized voice",
        "image_path": str(IMAGE_DIR / "andrew" / "andrew_neutral.jpg"),
        "customer_id": 100,
        "audio_path": str(DEMO_DIR / "audio" / "yes_approve.wav"),
        "expected": "denied",
    },
    {
        "name": "Full successful end-to-end path",
        "image_path": str(IMAGE_DIR / "andrew" / "andrew_neutral.jpg"),
        "customer_id": 100,
        "audio_path": str(AUDIO_DIR / "andrew" / "andrew_yes_approve.wav"),
        "expected": "granted",
    },
]

results = []
for scenario in scenarios:
    print(f"\n----- Scenario: {scenario['name']} -----")
    outcome = run_transaction(
        scenario["image_path"], scenario["customer_id"], scenario["audio_path"]
    )
    results.append({
        "scenario": scenario["name"],
        "expected": scenario["expected"],
        "got": outcome["status"],
        "passed": outcome["status"] == scenario["expected"],
    })

summary = pd.DataFrame(results)
print("\n=== Simulation Summary ===")
assert summary["passed"].all(), "One or more simulation scenarios failed"
summary


----- Scenario: Unauthorized image attempt -----
[Stage 1] Face check on '/Users/FullTimeStudio/Dev/Multimodal-Data-Preprocessing/formative2-multimodal-preprocessing/data/demo/images/unauthorized_face_01.jpg' -> authorized=False, member=None, confidence=0.00

*** ACCESS DENIED at Stage 1 (Face): unrecognized face (confidence=0.00) ***


----- Scenario: Authorized face, unauthorized voice -----
[Stage 1] Face check on '/Users/FullTimeStudio/Dev/Multimodal-Data-Preprocessing/formative2-multimodal-preprocessing/data/raw/images/andrew/andrew_neutral.jpg' -> authorized=True, member=andrew, confidence=0.98
[Stage 2] Product prediction computed silently (withheld until Stage 4).
[Stage 3] Voice check for claimed identity 'andrew' -> verified=False, confidence=0.00

*** ACCESS DENIED at Stage 3 (Voice): voice did not match (confidence=0.00) ***


----- Scenario: Full successful end-to-end path -----
[Stage 1] Face check on '/Users/FullTimeStudio/Dev/Multimodal-Data-Preprocessing/formative2-mu

,scenario,expected,got,passed
0,Unauthorized image attempt,denied,denied,True
1,"Authorized face, unauthorized voice",denied,denied,True
2,Full successful end-to-end path,granted,granted,True


## 5. How to re-run from the terminal

```bash
cd formative2-multimodal-preprocessing
python -m app.simulate_scenarios   # three demos + pass/fail table
python -m app.cli_app              # interactive prompts
```